# Deep research: plan, search, report, delivery: practice

Planner returns WebSearchPlan → searches run concurrently → writer returns ReportData → email agent delivers the report.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [2_openai/4_lab4.ipynb](../../../../../2_openai/4_lab4.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configure a real SDK agent and inspect its tool schema

Agent holds instructions/model/tools. The function_tool decorator constructs a schema from the signature and docstring. Constructing this configuration does not run the agent or contact a provider.

**Predict:** Which argument names will appear in the registered tool schema?

In [ ]:
from agents import Agent, function_tool, set_tracing_disabled
set_tracing_disabled(True)
@function_tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers."""
    return a + b
agent = Agent(name="Practice teacher", instructions="Use add_numbers for addition.",
              model="gpt-4o-mini", tools=[add_numbers])
print(agent.name, agent.tools[0].name)
print(sorted(agent.tools[0].params_json_schema["properties"]))
assert sorted(agent.tools[0].params_json_schema["properties"]) == ["a", "b"]

**Check:** a and b. The schema advertises the function; Runner executes the model/tool workflow when deliberately invoked.

**Change one thing:** Change the docstring and inspect the tool description. Find Runner.run in the source: that is where the interaction loop actually begins.

## 2. Plan → search → synthesize

The research workflow separates a search plan, collected findings, and a report. Use fixed fictional snippets first so you can inspect how evidence reaches the writer. No current facts are being researched here.

**Predict:** Will the report receive the original query as well as the findings?

In [ ]:
from dataclasses import dataclass
@dataclass
class SearchItem:
    reason: str
    query: str
query = "How should a beginner practice tool calls?"
plan = [SearchItem("Understand arguments", "tool argument schema"),
        SearchItem("Understand execution", "tool dispatcher")]
fixture = {"tool argument schema": "A schema describes arguments.",
           "tool dispatcher": "Application code executes an allowed function."}
findings = [fixture[item.query] for item in plan]
report = {"query": query, "summary": " ".join(findings), "follow_up_questions": ["How do we test invalid arguments?"]}
print(report)
assert len(findings) == 2 and "Application" in report["summary"]

**Check:** Yes. Plans choose questions, search collects material, and synthesis uses it. Delivery is a separate side effect.

**Change one thing:** Add a third search plan item and a matching fixture. Explain how the original query keeps synthesis focused.

## 3. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## 4. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## 5. Drafting and delivering are separate steps

An agent that formats text is different from a tool that sends email or notifications. This exercise records a delivery intention in memory only.

**Predict:** Will constructing this draft contact anyone?

In [ ]:
outbox = []
draft = {"to": "learner@example.com", "subject": "Practice report", "body": "Fictional findings."}
assert outbox == []
def dry_run_send(message):
    outbox.append({"status": "dry_run", "subject": message["subject"]})
    return {"status": "dry_run"}
result = dry_run_send(draft)
print(result, outbox)
assert len(outbox) == 1

**Check:** No. Neither this draft nor this in-memory dry run sends anything. A provider send/post call is the delivery boundary.

**Change one thing:** Format the body as HTML but keep dry_run_send unchanged. Explain where real delivery would begin.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.